Classify if Demented, Nondemented

In [ ]:
#set up a ColumnTransformer with StandardScaler for numerical features and OneHotEncoder for categorical features.
#set up and training a LinearRegression model using scikit-learn, including data preprocessing steps within a Pipeline.
#implement polynomial regression
#perform hyperparameter tuning for a polynomial regression model
#evaluate the performance of a regression model on test data
#use OneHotEncoder with handle_unknown='ignore' within a preprocessing pipeline to handle unseen categories during model training and evaluation
#set up and execute cross_val_score or GridSearchCV to perform cross-validation

In [3]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

# Load dataset
df = pd.read_csv('dementia.csv')

# Drop rows with missing target (MMSE)
df = df.dropna(subset=['MMSE'])

# Define target and features
target = 'MMSE'
X = df.drop(columns=[target, 'Subject ID', 'MRI ID'])  # Drop non-informative columns
y = df[target]

# Identify categorical and numerical columns
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()

# Preprocessing for numerical and categorical data
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features)
])

# Linear Regression pipeline
linear_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit linear regression
linear_pipeline.fit(X_train, y_train)
y_pred = linear_pipeline.predict(X_test)

# Evaluation
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Polynomial Regression pipeline
poly_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('poly', PolynomialFeatures()),
    ('regressor', LinearRegression())
])

# Grid search for best polynomial degree
param_grid = {
    'poly__degree': [1, 2, 3],
    'poly__include_bias': [False]
}

grid_search = GridSearchCV(poly_pipeline, param_grid, cv=5, scoring='r2')
grid_search.fit(X_train, y_train)

# Best model and predictions
best_model = grid_search.best_estimator_
y_poly_pred = best_model.predict(X_test)

# Evaluation of best polynomial model
poly_mse = mean_squared_error(y_test, y_poly_pred)
poly_r2 = r2_score(y_test, y_poly_pred)
cv_scores = cross_val_score(best_model, X, y, cv=5, scoring='r2')

# Results
results = {
    "Linear Regression": {"MSE": mse, "R2": r2},
    "Polynomial Regression": {
        "Best Degree": grid_search.best_params_['poly__degree'],
        "MSE": poly_mse,
        "R2": poly_r2,
        "CV Scores": cv_scores.tolist(),
        "Mean CV R2": np.mean(cv_scores)
    }
}

results


{'Linear Regression': {'MSE': 8.354445277283542, 'R2': 0.45514487322063846},
 'Polynomial Regression': {'Best Degree': 1,
  'MSE': 8.354445277283542,
  'R2': 0.45514487322063846,
  'CV Scores': [0.39722639188616315,
   0.397250881386331,
   0.4131314976329682,
   0.40323430222031764,
   0.5367400387555267],
  'Mean CV R2': np.float64(0.42951662237626137)}}